# Airbnb scraping: 10 European cities
Collects 2-bedroom "Entire home/apt" listings (first ~50 per city) and saves **only these columns**:

`city, room_id, name, total_price_eur, nights, rating, reviews, is_guest_favorite, is_superhost, url`

How to use: set `RUN_TEST = True` first (a few minutes), check the printed rows, then set `RUN_SCRAPING = True`
(about 1.5-2 hours). Both flags are `False` by default, so Run All does not open Chrome.

In [1]:
import re, time, random
from pathlib import Path
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import InvalidSessionIdException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

CITIES = ['Amsterdam', 'Athens', 'Barcelona', 'Berlin', 'Budapest',
          'Lisbon', 'London', 'Paris', 'Rome', 'Vienna']

CHECKIN, CHECKOUT = "2026-12-07", "2026-12-12"
TARGET = 50                  # listings per city

SKIP_BUILDINGS = True        # skip buildings with several units: their rating, reviews and price belong to the whole building
SKIP_NO_RATING = True        # skip listings that have no rating yet (new listings)

RUN_TEST     = False         # True: 5 listings in Barcelona + 3 in Rome, only prints, writes nothing
RUN_SCRAPING = False         # True: all cities (~1.5-2 hours), writes OUT_CSV

COLS = ["city", "room_id", "name", "total_price_eur", "nights",
        "rating", "reviews", "is_guest_favorite", "is_superhost", "url"]
OUT_CSV  = Path("Airbnb_Listings.csv")
PART_CSV = Path("Airbnb_Listings_partial.csv")     # progress file, deleted when everything is done


def build_url(city):
    return (f"https://www.airbnb.com/s/{city}/homes?query={city}&min_bedrooms=2"
            f"&room_types%5B%5D=Entire%20home%2Fapt"
            f"&checkin={CHECKIN}&checkout={CHECKOUT}&adults=2&currency=EUR")


def make_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def open_room(driver, room_id, retries=2):
    url = (f"https://www.airbnb.com/rooms/{room_id}"
           f"?adults=2&checkin={CHECKIN}&checkout={CHECKOUT}&currency=EUR")
    for attempt in range(retries + 1):
        try:
            driver.get(url)
            WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
            time.sleep(2)
            return
        except TimeoutException:
            if attempt == retries:
                raise

## Listing page: rating, reviews, Guest favorite, Superhost, single listing or building

In [2]:
def parse_reviews(body):
    """Returns (rating, reviews). (None, None) means an unknown page format."""
    m = re.search(r"(\d\.\d{1,2})\s*·\s*(\d[\d,]*)\s+reviews?", body, re.I)
    if not m:
        m = re.search(r"Rated\s+(\d\.\d{1,2})\s+out of 5 stars\.\s*[\d.]*\s*(\d[\d,]*)\s+reviews?",
                      body, re.I)
    if m:
        return float(m.group(1)), int(m.group(2).replace(",", ""))

    n = re.search(r"New\s*\n*\s*·\s*\n*\s*(\d[\d,]*)\s+reviews?", body, re.I)   # new listing with 1-2 reviews
    if n:
        return None, int(n.group(1).replace(",", ""))

    if re.search(r"No reviews \(yet\)", body) or re.search(r"New listing\s*\n\s*New\b", body):
        return None, 0
    return None, None


def classify_page(body):
    """'single', 'building:<rule>' or 'unknown'.
    A single listing has exactly one overview line like '6 guests · 2 bedrooms · 3 beds · 2 baths'.
    A building lists several units ('Choose your apartment', 'N apartments')."""
    if re.search(r"Choose your (?:apartment|unit)", body, re.I):
        return "building:choose"
    if re.search(r"(?m)^[ \t]*\d+\s+(?:apartments?|units?)[ \t]*$", body, re.I):
        return "building:units"
    n_overview = len(re.findall(r"(?m)^[ \t]*\d+\s+guests?\s*·", body))
    if n_overview >= 2:
        return "building:many-overview-lines"
    if n_overview == 0:
        return "unknown"
    return "single"


def page_text(driver):
    # innerText, not Selenium's .text: .text hides the "Superhost" line of the host block
    return driver.execute_script("return document.body.innerText")


def json_flags(driver):
    """(is_superhost, is_guest_favorite) from the data embedded in the page; None if not found.
    Only used as a cross-check of the text-based flags."""
    html = driver.page_source
    sh = re.findall(r'"isSuperhost":(true|false)', html)
    fav = re.findall(r'"isGuestFavorite":"?(true|false)"?', html)
    return ("true" in sh) if sh else None, ("true" in fav) if fav else None


def read_listing(driver):
    """Returns (details, kind, notes) for the listing page that is open.
    Reads the text first; scrolls only if rating/reviews are missing (they can be in a lazy block)."""
    h1 = driver.find_element(By.TAG_NAME, "h1").text
    body = page_text(driver)
    kind, (rating, reviews) = classify_page(body), parse_reviews(body)

    if kind == "unknown" or reviews is None:              # bottom blocks load lazily: scroll once, read again
        for _ in range(4):
            driver.execute_script("window.scrollBy(0, 1500);")
            time.sleep(0.7)
        body2 = page_text(driver)
        if kind == "unknown":
            kind = classify_page(body2)
        if reviews is None:
            rating, reviews = parse_reviews(body2)
        body = body + "\n" + body2                        # badges: look in both texts

    fav = bool(re.search(r"Guest\s+favorite", body))      # also matches the badge split over two lines
    host = "Superhost" in body
    notes = []
    js_host, js_fav = json_flags(driver)                  # cross-check with the page data
    if js_host is not None and js_host != host:
        notes.append(f"superhost: text says {host}, page data says {js_host}")
    if js_fav is not None and js_fav != fav:
        notes.append(f"guest_favorite: text says {fav}, page data says {js_fav}")
    if kind == "unknown":
        notes.append("page text starts with: " + repr(body[:300]))

    details = {"name": h1, "rating": rating, "reviews": reviews,
               "is_guest_favorite": int(fav), "is_superhost": int(host)}
    return details, kind, notes

## Search page: listing ids and prices

In [3]:
def find_next(driver, tries=3):
    """The 'Next' button can render late. Look for it a few times before deciding it is the last page."""
    for _ in range(tries):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        nxt = driver.find_elements(By.CSS_SELECTOR, "a[aria-label='Next']")
        if nxt:
            return nxt[0]
        time.sleep(3)
    return None


def collect_cards(driver):
    for _ in range(8):
        driver.execute_script("window.scrollBy(0, 1200);")
        time.sleep(1)

    cards = {}
    for card in driver.find_elements(By.CSS_SELECTOR, "div[data-testid='card-container']"):
        try:
            href = card.find_element(By.CSS_SELECTOR, "a[href*='/rooms/']").get_attribute("href")
            rid = re.search(r"/rooms/(\d+)", href).group(1)
        except Exception:
            continue
        text = " ".join((card.get_attribute("textContent") or "").split())
        m = re.search(r"€\s?([\d,]+)\s*for\s*(\d+)\s*nights?", text)
        cards[rid] = {
            "total_price_eur": int(m.group(1).replace(",", "")) if m else None,
            "nights": int(m.group(2)) if m else None,
        }
    return cards

## Scrape one city
Stage 1 collects ids and prices from the search pages (more than `target`, because some listings are skipped).
Stage 2 opens each listing until `target` valid rows are collected.

In [4]:
def scrape_city(driver, city, target=50, max_pages=8):
    # --- Stage 1: ids and prices from the search pages ---
    wanted = int(target * 2.5) if (SKIP_BUILDINGS or SKIP_NO_RATING) else target + 10
    driver.get(build_url(city))
    WebDriverWait(driver, 15).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div[data-testid='card-container']")))

    all_cards = {}
    for page in range(1, max_pages + 1):
        driver.execute_script("window.scrollTo(0, 0);")
        for rid, v in collect_cards(driver).items():
            all_cards.setdefault(rid, v)
        print(f"{city} - page {page}: {len(all_cards)} listings")
        if len(all_cards) >= wanted:
            break
        nxt = find_next(driver)
        if nxt is None:
            print("   no Next button after 3 tries, last page reached")
            break
        driver.execute_script("arguments[0].click();", nxt)
        time.sleep(4)

    # --- Stage 2: open each listing ---
    rows, skipped = [], {}

    def skip(reason, rid, show=True):
        skipped[reason] = skipped.get(reason, 0) + 1
        if show:
            print(f"   skipped {rid}: {reason}")

    for rid, card in all_cards.items():
        if len(rows) >= target:
            break
        if card["total_price_eur"] is None or card["nights"] is None:
            skip("no price on the search card", rid, show=False)
            continue
        try:
            open_room(driver, rid)
            d, kind, notes = read_listing(driver)
        except InvalidSessionIdException:
            raise                                          # browser is dead: stop
        except Exception as e:
            skip(f"error {type(e).__name__}", rid)
            continue

        for n in notes:
            print(f"   NOTE {rid} {n}")
        if kind.startswith("building") and SKIP_BUILDINGS:
            skip(kind, rid)
        elif kind == "unknown":
            skip("unknown page format", rid)
        elif SKIP_NO_RATING and d["rating"] is None:
            skip("no rating yet", rid, show=False)
        else:
            rows.append({"city": city, "room_id": rid, "name": d["name"],
                         "total_price_eur": card["total_price_eur"], "nights": card["nights"],
                         "rating": d["rating"], "reviews": d["reviews"],
                         "is_guest_favorite": d["is_guest_favorite"], "is_superhost": d["is_superhost"],
                         "url": f"https://www.airbnb.com/rooms/{rid}"})
            print(f"[{len(rows)}] {city} | €{card['total_price_eur']} ({card['nights']} nights) | "
                  f"★{d['rating']} ({d['reviews']}) | Fav:{d['is_guest_favorite']} SH:{d['is_superhost']}")
        time.sleep(random.uniform(1.0, 2.0))

    print(f"{city}: {len(rows)} rows | skipped: {skipped or 'none'}")
    if len(rows) < target:
        print(f"   WARNING only {len(rows)}/{target} rows for {city}")
    return rows, skipped

## Test run (a few minutes)
Look at the printed rows and at the `skipped` reasons before starting the full run.

In [5]:
if RUN_TEST:
    driver = make_driver()
    try:
        for city, n in [("Barcelona", 5), ("Rome", 3)]:
            rows, skipped = scrape_city(driver, city, target=n)
            print(pd.DataFrame(rows, columns=COLS).drop(columns=["url"]).to_string(), "\n")
    finally:
        driver.quit()

## Full run (about 1.5-2 hours)
Progress is saved after every city in `Airbnb_Listings_partial.csv`. If it stops, run the cell again:
finished cities are skipped. The final file has exactly the columns in `COLS`.

In [6]:
if RUN_SCRAPING:
    all_rows, done_cities = [], set()
    if PART_CSV.exists():
        part = pd.read_csv(PART_CSV, dtype={"room_id": str})
        all_rows, done_cities = part.to_dict("records"), set(part["city"])
        print("resuming, finished cities:", sorted(done_cities))

    driver = make_driver()
    try:
        for city in CITIES:
            if city in done_cities:
                continue
            rows, skipped = scrape_city(driver, city, target=TARGET)
            all_rows.extend(rows)
            pd.DataFrame(all_rows, columns=COLS).to_csv(PART_CSV, index=False)
            print(f"OK {city}: {len(rows)} | total: {len(all_rows)}\n")
    finally:
        try:
            driver.quit()
        except Exception:
            pass

    df = pd.DataFrame(all_rows, columns=COLS)
    df["room_id"] = df["room_id"].astype(str)
    for c in ["total_price_eur", "nights", "reviews"]:
        df[c] = df[c].astype("Int64")                     # whole numbers, no "5.0"
    df.to_csv(OUT_CSV, index=False, encoding="utf-8")
    PART_CSV.unlink(missing_ok=True)
    print("saved", OUT_CSV, df.shape)

## Re-scrape some cities (optional)
If a city came back incomplete, put its name in `FIX_CITIES`. The other cities in the file are kept,
and a safety copy of the file is saved first. Leave the list empty to skip this cell.

## Final check

In [7]:
df = pd.read_csv(OUT_CSV, dtype={"room_id": str})

assert list(df.columns) == COLS
assert df["room_id"].is_unique
assert df["total_price_eur"].notna().all() and (df["total_price_eur"] > 0).all()
assert df["nights"].notna().all() and df["nights"].between(1, 30).all()
assert df["rating"].dropna().between(1, 5).all()
assert (df["reviews"].dropna() >= 0).all()
assert df[["is_guest_favorite", "is_superhost"]].isin([0, 1]).all().all()

print(df.shape)
print(df.groupby("city").size())
print("nulls:", df.isna().sum()[lambda s: s > 0].to_dict())
print("nights:", df["nights"].value_counts().sort_index().to_dict())
df.head()

(500, 10)
city
Amsterdam    50
Athens       50
Barcelona    50
Berlin       50
Budapest     50
Lisbon       50
London       50
Paris        50
Rome         50
Vienna       50
dtype: int64
nulls: {}
nights: {3: 19, 4: 30, 5: 444, 6: 1, 7: 6}


,city,room_id,name,total_price_eur,nights,rating,reviews,is_guest_favorite,is_superhost,url
0,Amsterdam,30762441,Stylish Serviced Apartment near Central Station,1596,5,4.81,1769,0,1,https://www.airbnb.com/rooms/30762441
1,Amsterdam,1113044804402014015,Numa | 2 Bedroom Apartment in Amsterdam East,1115,5,4.56,233,0,0,https://www.airbnb.com/rooms/1113044804402014015
2,Amsterdam,1793971,NoorderPark,820,5,4.86,373,1,1,https://www.airbnb.com/rooms/1793971
3,Amsterdam,33092943,"2-Bedroom with Bath, Sleeps 6, Amstelveen, 75m2",822,5,4.78,809,0,1,https://www.airbnb.com/rooms/33092943
4,Amsterdam,953539337193788533,Stylish Riverfront Apartment with Stunning Views,1292,5,4.73,254,0,0,https://www.airbnb.com/rooms/953539337193788533


## Limitations
- Sample = first ~50 valid results per city in Airbnb's default ranking, not random.
- Requested dates 7-12 Dec 2026 for 2 guests; only listings available for those dates appear.
- `total_price_eur` is the total including fees for `nights` nights (not always 5): compare listings per night.
- Buildings with several units and listings without a rating are skipped (switches at the top).
- Superhost / Guest favorite are read from page text and were only checked by eye on a small sample.
- Educational use. Check Airbnb's terms before publishing the data.